# Lab 02 — Tuning and evaluating retrieval

**Level:** Intermediate · **Duration:** ~60 minutes · **Covers:** H3 chunking strategy comparison: fixed vs semantic (30 min) · H4 retrieval quality evaluation (30 min)

Lab guide: [`../lab-02-tuning-evaluating-retrieval.md`](../lab-02-tuning-evaluating-retrieval.md) — read each part there first; this notebook holds the code.

Advanced RAG in LlamaIndex starts with measurement. You write the retrieval metrics (TODO), then use them to compare chunking strategies and retrievers on 24 labelled Daimler Truck questions, and judge answer relevance with an LLM on both providers.

In [ ]:
# (notebook) Setup: run this cell first. LLM_PROVIDER picks the model, EMBEDDING_PROVIDER the embeddings.
import json
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # Chroma: no usage telemetry
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_colwidth", 110)

# labs/module-03 holds data/ and rag_kit.py; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-03" and (p / "data").is_dir())
DATA_DIR, OUTPUT_DIR = LAB_DIR / "data", LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path[:0] = [str(LAB_DIR.parent / "setup"), str(LAB_DIR)]
import rag_kit as rk  # noqa: E402
from llm import (  # noqa: E402
    describe, embedding_slug, get_embedding_model_name, get_embedding_provider, get_provider,
    is_configured, partner_provider,
)
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.disable_progress_bar()
PROVIDER, PARTNER = get_provider(), partner_provider()
if not is_configured(PROVIDER):
    raise SystemExit(
        f"❌ LLM_PROVIDER={PROVIDER} is not configured. Add its key and model to the repo-root .env "
        "(template: labs/setup/.env.example), restart the kernel and run this cell again."
    )
HAS_PARTNER = is_configured(PARTNER)


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(describe())
print(f"Comparison provider: {PARTNER} {'✅' if HAS_PARTNER else '⚠️  not configured: comparison cells skip it'}")
print(f"Embeddings: {get_embedding_provider()} · {get_embedding_model_name()} (index suffix {embedding_slug()})")

from llama_index.core import PromptTemplate, QueryBundle, Settings
from llama_index.core.node_parser import MarkdownNodeParser, SemanticSplitterNodeParser, SentenceSplitter
from llama_index.core.query_engine import RetrieverQueryEngine

rk.setup_llama(PROVIDER)
QUERIES = json.loads((DATA_DIR / "test_queries.json").read_text(encoding="utf-8"))
li_docs = rk.to_llama_documents(rk.load_corpus())
K = 4

## Part 1 — Labelled queries and retrieval metrics (10 min)

Each query lists the documents that answer it. The set mixes fault codes and part numbers,
paraphrases, German and cross-lingual questions, and three questions the corpus cannot answer.

In [ ]:
# (notebook) The labelled set: 24 answerable queries and 3 unanswerable ones.
print(pd.Series([q["kind"] for q in QUERIES]).value_counts().to_dict())
pd.DataFrame(QUERIES).head(8)

In [ ]:
# (notebook) Your TODO: the four retrieval metrics for ONE query. Relevance is judged per document.
def retrieval_metrics(ids: list[str], relevant: set[str], k: int) -> dict[str, float]:
    """Metrics for the top k of `ids` (doc_ids in rank order; one entry per retrieved chunk, so an id can repeat).

    - precision: relevant entries in the top k / k
    - recall:    distinct relevant documents found in the top k / number of relevant documents
    - hit_rate:  1.0 if any relevant document is in the top k, else 0.0
    - mrr:       1 / rank of the first relevant entry (ranks start at 1), or 0.0 if none
    """
    # TODO: compute and return the four metrics as a dict with exactly these keys.
    raise NotImplementedError("retrieval_metrics")

In [ ]:
# (notebook) The harness: mean metrics over the answerable queries. It uses your function once it works.
metrics = rk.todo_or_reference(retrieval_metrics, rk.retrieval_metrics, ["KB-0610"], {"KB-0610"}, 1)


def evaluate(retrieve_ids, queries=QUERIES, k: int = K) -> dict[str, float]:
    rows = [metrics(retrieve_ids(q["query"]), set(q["relevant"]), k) for q in queries if q["relevant"]]
    return {m: round(sum(r[m] for r in rows) / len(rows), 2) for m in rows[0]}

In [ ]:
# (notebook) Check: the metrics on cases you can work out by hand.
def check_part1():
    m = retrieval_metrics(["A", "B", "A", "C"], {"A", "D"}, 4)
    assert m == {"precision": 0.5, "recall": 0.5, "hit_rate": 1.0, "mrr": 1.0}, m
    m = retrieval_metrics(["B", "C", "D", "E", "A"], {"D"}, 4)
    assert m["mrr"] == 1 / 3 and m["hit_rate"] == 1.0 and m["precision"] == 0.25, m
    assert retrieval_metrics(["B", "C"], {"A"}, 4) == {"precision": 0.0, "recall": 0.0, "hit_rate": 0.0, "mrr": 0.0}


check("Part 1 — retrieval metrics", check_part1)

## Part 2 — Fixed vs semantic chunking (20 min)

Same documents, same embedding model, same retriever (dense, top 4). Only the node parser changes.
Each strategy gets its own Chroma collection.

In [ ]:
# (notebook) Four node parsers. SentenceSplitter sizes are tokens; the semantic splitter cuts where meaning shifts.
# Pass embed_model: SemanticSplitterNodeParser.from_defaults() falls back to OpenAIEmbedding(), not Settings.embed_model.
SPLITTERS = {
    "fixed-128": SentenceSplitter(chunk_size=128, chunk_overlap=16),
    "fixed-512": SentenceSplitter(chunk_size=512, chunk_overlap=50),
    "semantic": SemanticSplitterNodeParser.from_defaults(embed_model=Settings.embed_model, breakpoint_percentile_threshold=90),
    "markdown": MarkdownNodeParser(),  # structure-aware: one node per heading section
}
indexes, rows = {}, []
for name, splitter in SPLITTERS.items():
    nodes = splitter.get_nodes_from_documents(li_docs)
    index = rk.build_index(nodes, f"lab02-{name}-{embedding_slug()}")
    retriever = index.as_retriever(similarity_top_k=K)
    indexes[name] = (index, nodes)
    rows.append({"strategy": name, "nodes": len(nodes),
                 "median chars": int(pd.Series([len(n.get_content()) for n in nodes]).median()),
                 **evaluate(lambda q, r=retriever: rk.doc_ids(r.retrieve(q)))})
chunking = pd.DataFrame(rows).set_index("strategy")
chunking

In [ ]:
# (notebook) Read what each strategy actually hands the model for one question.
q = "How long does a DC fast charge from 20 to 80 percent take on an ED-26?"
for name, (index, _) in indexes.items():
    top = index.as_retriever(similarity_top_k=1).retrieve(q)[0]
    print(f"--- {name} · {top.node.metadata['doc_id']} · {len(top.node.get_content())} chars\n{top.node.get_content()[:300]}\n")

📝 Copy the table into [`templates/chunking-workbook.md`](../templates/chunking-workbook.md) and
answer its questions: which strategy has the best precision, and would you ship it?

## Part 3 — Keyword, hybrid and re-ranked retrieval (15 min)

Vector search matches meaning; BM25 matches exact tokens such as `FC-BRK-0705` and `PN 47-0118-341`.
Hybrid retrieval fuses both rankings; a cross-encoder then re-reads the best candidates.

In [ ]:
# (notebook) Four retrievers over the markdown-section nodes.
index, nodes = indexes["markdown"]
dense = index.as_retriever(similarity_top_k=K)
bm25 = rk.bm25_retriever(nodes, K)
hybrid = rk.hybrid_retriever(index, nodes, k=K, pool=10)  # reciprocal rank fusion, no LLM
rerank = rk.reranker(K)                                   # RERANK_MODEL, a local cross-encoder


def hybrid_rerank_ids(query: str) -> list[str]:
    candidates = rk.hybrid_retriever(index, nodes, k=10, pool=10).retrieve(query)
    return rk.doc_ids(rerank.postprocess_nodes(candidates, query_bundle=QueryBundle(query)))


RETRIEVERS = {
    "dense": lambda q: rk.doc_ids(dense.retrieve(q)),
    "bm25": lambda q: rk.doc_ids(bm25.retrieve(q)),
    "hybrid": lambda q: rk.doc_ids(hybrid.retrieve(q)),
    "hybrid + rerank": hybrid_rerank_ids,
}
pd.DataFrame({name: evaluate(fn) for name, fn in RETRIEVERS.items()}).T

In [ ]:
# (notebook) The averages hide the story: which queries does each retriever miss?
hits = pd.DataFrame({
    name: {q["qid"]: metrics(fn(q["query"]), set(q["relevant"]), K)["hit_rate"] for q in QUERIES if q["relevant"]}
    for name, fn in RETRIEVERS.items()
})
misses = hits[hits.min(axis=1) < 1].join(pd.DataFrame(QUERIES).set_index("qid")[["query", "kind"]])
misses

In [ ]:
# (notebook) Check: hybrid retrieval finds at least as many queries as dense retrieval alone.
def check_part3():
    assert hits["hybrid"].sum() >= hits["dense"].sum(), hits.sum()
    assert evaluate(RETRIEVERS["hybrid"])["recall"] >= 0.9


check("Part 3 — hybrid retrieval", check_part3)

## Part 4 — Answer relevance with an LLM judge, on both providers (15 min)

Retrieval metrics say whether the right documents came back; a judge says whether the **answer**
addresses the question. Unanswerable questions are scored separately: there, a refusal is the pass.

In [ ]:
# (notebook) Answer five questions with the hybrid retriever (two of them unanswerable).
from pydantic import BaseModel, Field
from typing import Literal

QA_PROMPT = PromptTemplate(
    "Answer only from the sources and cite the doc_id in square brackets, for example [KB-0610]. "
    "If the sources do not answer the question, say so.\n\n<sources>\n{context_str}\n</sources>\n\n"
    "Question: {query_str}\nAnswer: "
)
engine = RetrieverQueryEngine.from_args(hybrid, text_qa_template=QA_PROMPT)
EVAL_SET = [q for q in QUERIES if q["qid"] in {"q05", "q08", "q10", "u01", "u02"}]
answers = {q["qid"]: str(engine.query(q["query"])) for q in EVAL_SET}
for q in EVAL_SET:
    print(f"{q['qid']} {q['query']}\n    -> {answers[q['qid']][:160]}\n")

In [ ]:
# (notebook) A small-tier judge with a structured verdict, on both providers.
class Relevance(BaseModel):
    score: Literal[1, 2, 3, 4, 5] = Field(description="5 = fully addresses the question, 1 = not at all")
    refused: bool = Field(description="True if the answer says the information is not available")
    reason: str = Field(description="One sentence")


JUDGE = PromptTemplate(
    "You grade answer relevance: does the ANSWER address the QUESTION that was asked? Ignore whether it is "
    "true; groundedness is graded separately.\n\nQUESTION: {question}\n\nANSWER: {answer}"
)
judged = []
for provider in [PROVIDER, PARTNER] if HAS_PARTNER else [PROVIDER]:
    judge = rk.llama_llm(provider, small=True)
    for q in EVAL_SET:
        verdict = judge.structured_predict(Relevance, JUDGE, question=q["query"], answer=answers[q["qid"]])
        passed = verdict.refused if not q["relevant"] else verdict.score >= 4
        judged.append({"judge": provider, "qid": q["qid"], "answerable": bool(q["relevant"]),
                       "score": verdict.score, "refused": verdict.refused, "pass": passed})
judged = pd.DataFrame(judged)
judged.pivot_table(index="qid", columns="judge", values=["score", "pass"], aggfunc="first")

In [ ]:
# (notebook) Check: every answer was judged, and answerable and unanswerable questions are scored apart.
def check_part4():
    assert len(judged) == len(EVAL_SET) * judged["judge"].nunique()
    assert set(judged["answerable"]) == {True, False}


check("Part 4 — answer relevance judged", check_part4)

📝 **Write-up (3–5 sentences).** Fill in [`templates/retrieval-eval-rubric.md`](../templates/retrieval-eval-rubric.md)
for the configuration you would ship (chunking, retriever, re-ranker or not, k). Which query did the
re-ranker lose, and why? Did the two judges agree?

> *Your answer here.*

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 03 — Enterprise connectors and use-case assistants](../lab-03-enterprise-connectors-assistants.md).